<a href="https://colab.research.google.com/github/MJ-Zu/wmam/blob/colab/notebooks/colab_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# wmam on Colab (GPU)

Runtime → Change runtime type → **GPU** before running. Every training command takes only a
**setup file** (input) and an **output path**:

```
python -m kinkchain.make_setup --config configs/example_cyclic.json --out setups/example_cyclic.setup.pkl   # freeze data + init + kinks
python -m kinkchain.run_setup  --setup setups/example_cyclic.setup.pkl         --out runs/example_bulk.npz       # train
```

Code comes from GitHub (`git clone` / `git pull`); outputs go to `runs/` (not committed) — here
optionally a Google-Drive folder so they survive the session.

## 1. GPU check

In [1]:
# run in a subprocess: a JAX client inside the notebook process would grab most of the GPU memory
!nvidia-smi -L
!python -c "import jax; print('jax', jax.__version__, jax.devices())"

GPU 0: Tesla T4 (UUID: GPU-2cb93059-3afd-3d8a-4446-c2e4c3f88346)
jax 0.11.1 [CudaDevice(id=0)]


## 2. Code from GitHub

In [58]:
import os

REPO_URL = "https://github.com/MJ-Zu/wmam.git"   # public: no login needed
BRANCH   = "main"
WORKDIR  = "/content/wmam"

if not os.path.isdir(WORKDIR):
    !git clone --branch {BRANCH} {REPO_URL} {WORKDIR}
else:
    !git -C {WORKDIR} pull --ff-only       # pick up what you pushed since the last session
%cd {WORKDIR}
!git log --oneline -3

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.66 MiB | 9.12 MiB/s, done.
From https://github.com/MJ-Zu/wmam
   5293eb2..2faf286  main       -> origin/main
Updating 5293eb2..2faf286
Fast-forward
 data/disk_data_1000_1500_strat.npz | Bin 0 -> 2572528 bytes
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 data/disk_data_1000_1500_strat.npz
/content/wmam
2faf286 (HEAD -> main, origin/main, origin/HEAD) add new data set
5293eb2 Setup files via JSON/pickle; run_setup; Colab notebook
6159ab8 Initial commit: single-tau tiny CNN training with kinked reparameterization


In [3]:
# jax comes with Colab (CUDA build on a GPU runtime); the rest is small
!pip install -q -U flax optax
# if the GPU check above showed only CpuDevice:  !pip install -q -U "jax[cuda12]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.5/532.5 kB 10.5 MB/s eta 0:00:00


## 3. Where the outputs go (optional: Google Drive)

`runs/` becomes a link to a Drive folder, so every output written by `--out runs/...` lands on Drive
directly and survives a disconnected runtime. Set `USE_DRIVE = False` to keep them on the
(temporary) Colab disk and download at the end.

In [4]:
import os, shutil

USE_DRIVE  = True
DRIVE_RUNS = "/content/drive/MyDrive/wmam_runs"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_RUNS, exist_ok=True)
    if os.path.isdir("runs") and not os.path.islink("runs"):
        shutil.copytree("runs", DRIVE_RUNS, dirs_exist_ok=True)
        shutil.rmtree("runs")
    if not os.path.lexists("runs"):
        os.symlink(DRIVE_RUNS, "runs")
os.makedirs("runs", exist_ok=True)
os.makedirs("setups", exist_ok=True)
print(os.path.realpath("runs"))

Mounted at /content/drive
/content/drive/MyDrive/wmam_runs


## 4. Look at a config

A config JSON is `{"cfg": {...}, "run": {...}}`: `cfg` = WMAMConfig fields (the rest are defaults),
`run` = `mode` (`cyclic` | `single_tau` | `refine` | `further`), `grad_clip`, `stride`, ...
Edit a file under `configs/` (or push the change to GitHub and `git pull`), or just edit the dict
here and write it out.

In [6]:


CONFIG = "configs/example_cyclic.json"     # try configs/example_utkface_cyclic.json for the 64x64 faces
print(json.dumps(json.load(open(CONFIG)), indent=1))

{
 "cfg": {
  "data": "data/disk_data_100_150_strat.npz",
  "img": 16,
  "n_train": 100,
  "n_test": 150,
  "channels": 6,
  "kink_fraction": 0.4,
  "s_low": 50.0,
  "s_high": 50.0,
  "seed": 27140,
  "tau_min": 0.5,
  "tau_max": 0.55,
  "increment": 0.01,
  "near_band": 0.12,
  "bulk_optimizer": "adam",
  "bulk_lr": 0.001,
  "bulk_max_steps": 500,
  "bulk_cycles": 30,
  "refine_max_steps": 1000
 },
 "run": {
  "mode": "cyclic",
  "grad_clip": 10.0,
  "stride": 1
 }
}


Memory: the per-step weight trajectory takes `(steps + 1) × n_parameters × 4` bytes on the GPU
(64 channels, 100 000 steps ≈ 14 GiB). Reduce it with `"stride": k` (cyclic modes) or `"log_every": k`
(single_tau) in the config — only every k-th step is recorded, *inside* the training loop.

## Single train

Single fixed τ (bulk → refine) uses a `single_tau` config:
`!python -m kinkchain.run_setup --setup configs/example_single_tau.json --out runs/example_single_tau.npz`

In [60]:
import json

CONFIG = "configs/example_single_tau.json"     # try configs/example_utkface_cyclic.json for the 64x64 faces
print(json.dumps(json.load(open(CONFIG)), indent=1))

{
 "cfg": {
  "data": "data/disk_data_100_150_strat.npz",
  "img": 16,
  "channels": 6,
  "kink_fraction": 0.4,
  "s_low": 100.0,
  "s_high": 100.0,
  "seed": 11658,
  "seed_kink": 24231,
  "tau_max": 0.55,
  "bulk_optimizer": "gd",
  "bulk_lr": 0.0001,
  "bulk_max_steps": 10000,
  "bulk_tol": 1e-10,
  "refine_optimizer": "gd",
  "refine_lr": 3e-05,
  "refine_max_steps": 1000,
  "refine_tol": 1e-10,
  "log_every": 10
 },
 "run": {
  "mode": "single_tau",
  "tau": 0.55,
  "grad_clip": 100.0
 }
}


In [63]:
# Load the configuration file as a dictionary instead of just copying the filename string
CONFIG = "configs/example_single_tau.json"
job = json.load(open(CONFIG))

job["cfg"].update(
    data="data/disk_data_1000_1500_strat.npz",
    n_train=1000,
    n_test=1500,
    channels=24,
    bulk_optimizer="adam",
    bulk_lr=1e-3,
    bulk_max_steps=10_000,
    refine_max_steps=1,
    refine_lr=3e-4
)
job["run"].update(mode="single_tau", tau=0.53, grad_clip=100.0)

CONFIG = "configs/single_tau0.53_run.json"
json.dump(job, open(CONFIG, "w"), indent=1)

In [64]:
!python -m kinkchain.run_setup \
--setup configs/single_tau0.53_run.json \
--out runs/single_tau0.53_kink0.4.npz

[bulk   ] steps=10000  final_loss=5.774e-07  n_thetas=  5473  final_test_loss=4.789e-02  test_acc=0.993  near_tau_acc(|a-0.53|<0.12)=0.967 (n=336)
[refine ] steps=    1  final_loss=5.774e-07  n_thetas=  5473  final_test_loss=4.789e-02  test_acc=0.993  near_tau_acc(|a-0.53|<0.12)=0.967 (n=336)
saved -> runs/single_tau0.53_kink0.4.npz


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axs = plt.subplots(2, 1, figsize=(6,8))

log = np.load("runs/single_tau0.53_kink0.4.npz")
loss = log["bulk_loss"]
val_loss = log["bulk_test_loss"]
# # refine_loss12 = log12["refine_loss"]
steps = np.arange(len(loss))
theta_traj = log["bulk_theta"]
kink = log["kink"]

axs[0].plot(steps, loss, c="gray")
axs[0].plot(steps, val_loss, '--', c="gray")

kink_idx = np.where(kink)[0]
colors = plt.cm.viridis(np.linspace(0, 1, max(len(kink_idx), 1)))
# for c, j in enumerate(kink_idx):
#     axs[1].plot(steps2, np.abs(theta_traj2[:, j]), color=colors[c], lw=1.0, zorder=2,
#                 label=f"w[{j}]")

axs[1].plot(steps, theta_traj[:,:10], color="0.5", lw=1.0, zorder=2)

axs[0].set_yscale('log')
axs[0].set_xscale('log')


# axs[0].set_xlim(2, 100)
# axs[0].set_ylim(1.e-3, 1.e-1)
axs[0].set_ylabel("loss")
axs[0].set_xlabel("steps")

axs[1].set_yscale('log')
axs[1].set_xscale('log')
# axs[1].set_xlim(1e4, 2e4)
# axs[1].set_ylim(-1.e-4, 1.e-4)
axs[1].set_ylabel("d_theta")
axs[1].set_xlabel("steps")

axs[0].legend(loc='best')


## 5. Freeze the setup, then train

In [59]:
# Load the configuration file as a dictionary instead of just copying the filename string
CONFIG = "configs/example_cyclic.json"
job = json.load(open(CONFIG))

job["cfg"].update(
    data="data/disk_data_1000_1500_strat.npz",
    n_train=1000,
    n_test=1500,
    s_low=100.0,
    s_high=100.0,
    channels=24,
    seed=1001,
    bulk_lr=1e-4,
    bulk_max_steps=500,
    bulk_cycles=10,
)
job["run"].update(stride=20, grad_clip=10.0)

CONFIG = "configs/my_run.json"
json.dump(job, open(CONFIG, "w"), indent=1)

In [41]:
!python -m kinkchain.make_setup --config {CONFIG} --out setups/channels24_cyclic.setup.pkl
!python -m kinkchain.make_setup --show setups/channels24_cyclic.setup.pkl

froze configs/my_run.json -> setups/channels24_cyclic.setup.pkl  (mode=cyclic, 5473 parameters, 2184 kinked, 100 train / 150 test images)
setups/channels24_cyclic.setup.pkl: version 1, kind cyclic
run: {"batch_seed": 12345, "grad_clip": 10.0, "load": null, "mode": "cyclic", "out": null, "rescale": false, "stride": 20, "tau": null}
cfg: {"augment": false, "batch_size": 0, "budget_steps": 500, "bulk_afactor": 0.3, "bulk_cycles": 10, "bulk_lr": 0.0001, "bulk_max_steps": 500, "bulk_optimizer": "adam", "bulk_tol": 1e-11, "channels": 24, "converge_tol": 1e-11, "dD_patience": 3, "dD_plateau_rel": 0.01, "data": "data/disk_data_100_150_strat.npz", "img": 16, "increment": 0.01, "kink_fraction": 0.4, "n_test": 150, "n_train": 100, "near_band": 0.12, "property_noise": 0.0, "readout_lr": 0.0001, "readout_max_steps": 1000, "readout_tol": 1e-10, "refine_afactor": 1.0, "refine_cycles": 100, "refine_lr": 0.0001, "refine_max_steps": 1000, "refine_optimizer": "gd", "refine_tol": 1e-10, "s_high": 50.0, "s

In [42]:
# stage 1 (cyclic: cold + bulk cycles)
!python -m kinkchain.run_setup --setup setups/channels24_cyclic.setup.pkl --out runs/channels24_bulk.npz

[np.float64(0.54), np.float64(0.53), np.float64(0.52), np.float64(0.51), np.float64(0.5)]
[np.float64(0.51), np.float64(0.52), np.float64(0.53), np.float64(0.54), np.float64(0.55)]
loss_cold=5.112473e+01
  bulk   1  dD=6.31e-03  max_loss=2.13e-06  converged=False  knot_crossings=133  valid_loss=1.68e+00
  bulk   3  dD=6.37e-04  max_loss=1.66e-12  converged=True  knot_crossings=47  valid_loss=1.76e+00
  bulk   6  dD=3.10e-04  max_loss=5.01e-16  converged=True  knot_crossings=42  valid_loss=1.94e+00
  bulk   9  dD=3.01e-04  max_loss=2.90e-15  converged=True  knot_crossings=36  valid_loss=1.41e+00
[cyclic] 330 steps logged -> runs/channels24_bulk.npz


Next stages are small JSONs layered on the frozen setup (`"base": "setups/example_cyclic.setup.pkl"`), see
`configs/example_stage2_further.json` and `configs/example_stage3_refine.json` — their
`run.load` names the previous output. (`scripts/cyclic_pipeline.sh` runs all three in a row.)

In [29]:
CONFIG = "configs/example_stage2_further.json"     # try configs/example_utkface_cyclic.json for the 64x64 faces
print(json.dumps(json.load(open(CONFIG)), indent=1))

{
 "base": "setups/example_cyclic.setup.pkl",
 "cfg": {
  "refine_optimizer": "adam",
  "refine_lr": 0.0003,
  "refine_afactor": 1.0,
  "refine_max_steps": 500,
  "refine_cycles": 100
 },
 "run": {
  "mode": "further",
  "load": "runs/example_bulk.npz",
  "grad_clip": 100.0
 }
}


In [50]:
job = json.load(open(CONFIG))

# Corrected: Directly assign the string path to "base" instead of calling .update()
job["base"] = "setups/channels24_cyclic.setup.pkl"
job["cfg"].update(
    refine_lr=1e-6,
    refine_max_steps=1000,
    refine_cycles=30,
)
job["run"].update(
    mode="further",
    load="runs/channels24_bulk.npz",
    stride=20,
    grad_clip=100.0)   # stride 控制显存

CONFIG = "configs/my_run_stage2.json"                 # 之后的单元用改过的副本
json.dump(job, open(CONFIG, "w"), indent=1)

# Stage 2 config does not go through make_setup; it is executed directly via run_setup
print(f"Config written to {CONFIG}. Run this with run_setup in the next cell.")

Config written to configs/my_run_stage2.json. Run this with run_setup in the next cell.


In [51]:
!python -m kinkchain.run_setup \
  --setup configs/my_run_stage2.json \
  --out runs/channels24_further.npz \
  --load runs/channels24_bulk.npz
# !python -m kinkchain.run_setup --setup configs/example_stage3_refine.json  --out runs/example_refine.npz

further bulk cyclic training
[refine] recalled theta_max/theta_min from runs/channels24_bulk.npz
  refine  1  dD=1.55e-04  max_loss=9.91e-08    knot_crossing=1  valid_loss=1.50e+00
  refine  2  dD=2.91e-06  max_loss=1.02e-07    knot_crossing=0  valid_loss=1.49e+00
  refine  3  dD=2.89e-06  max_loss=1.02e-07    knot_crossing=0  valid_loss=1.48e+00
  refine  4  dD=2.86e-06  max_loss=1.04e-07    knot_crossing=1  valid_loss=1.47e+00
  refine  5  dD=5.26e-06  max_loss=6.21e-10    knot_crossing=1  valid_loss=1.51e+00
  refine  6  dD=2.69e-06  max_loss=1.33e-12    knot_crossing=0  valid_loss=1.68e+00
  -> MAM approached at refine cycle 5
[cyclic] 921 steps logged -> runs/channels24_further.npz


In [53]:
CONFIG = "configs/example_stage3_refine.json"     # try configs/example_utkface_cyclic.json for the 64x64 faces
print(json.dumps(json.load(open(CONFIG)), indent=1))

{
 "base": "setups/example_cyclic.setup.pkl",
 "cfg": {
  "refine_optimizer": "gd",
  "refine_lr": 4e-06,
  "refine_afactor": 1.0,
  "refine_max_steps": 3000,
  "refine_tol": 1e-07,
  "refine_cycles": 100
 },
 "run": {
  "mode": "refine",
  "load": "runs/example_further.npz",
  "grad_clip": 100.0
 }
}


In [57]:
job = json.load(open(CONFIG))

# Corrected: Directly assign the string path to "base" instead of calling .update()
job["base"] = "setups/channels24_cyclic.setup.pkl"
job["cfg"].update(
    refine_optimizer="gd",
    refine_lr=1e-6,
    refine_tol=1e-7,
    refine_max_steps=2000,
    refine_cycles=10,
)
job["run"].update(
    mode="refine",
    load="runs/channels24_further.npz",
    stride=20,
    grad_clip=100.0)   # stride 控制显存

CONFIG = "configs/my_run_stage3.json"                 # 之后的单元用改过的副本
json.dump(job, open(CONFIG, "w"), indent=1)

# Stage 2 config does not go through make_setup; it is executed directly via run_setup
print(f"Config written to {CONFIG}. Run this with run_setup in the next cell.")

!python -m kinkchain.run_setup \
  --setup configs/my_run_stage3.json \
  --out runs/channels24_refine.npz \
  --load runs/channels24_further.npz

Config written to configs/my_run_stage3.json. Run this with run_setup in the next cell.
0.4 1e-06 1e-07 0 2000
[refine] recalled theta_max/theta_min from runs/channels24_further.npz
  refine  1  dD=1.65e-06  max_loss=1.00e-07    knot_crossing=0  valid_loss=1.77e+00
  -> MAM approached at refine cycle 0
[cyclic] 124 steps logged -> runs/channels24_refine.npz


Single fixed τ (bulk → refine) uses a `single_tau` config:
`!python -m kinkchain.run_setup --setup configs/example_single_tau.json --out runs/example_single_tau.npz`

## 6. Outputs

In [ ]:
!ls -lh runs

In [ ]:
# without Drive: download everything in runs/ as one zip
if not USE_DRIVE:
    shutil.make_archive("/content/wmam_runs", "zip", "runs")
    from google.colab import files
    files.download("/content/wmam_runs.zip")